# Clipt Video Upscaling — Real-ESRGAN

Upscales detected clip segments from 360p to 720p using AI super-resolution.

**Runtime:** Google Colab with A100 GPU. Set this in `Runtime > Change runtime type > Hardware accelerator > GPU > A100`.

**Inputs:** YouTube URL or Cloudinary URL of the full game film + a clip manifest (startTime/endTime list).

**Outputs:** Each clip upscaled 2x (640x360 → 1280x720) and uploaded to Cloudinary, with a JSON manifest saved to Google Drive.

**Expected wall time:** 5-10 minutes for ~10 clips on A100.

## Cell 1 — Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU — enable A100 in Runtime > Change runtime type'
print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0))
print('Torch:', torch.__version__)

## Cell 2 — Install Real-ESRGAN + dependencies

Pins to known-good versions for Colab CUDA 12. The `setup.py develop` step makes the inference scripts importable.

In [ ]:
%cd /content
!pip install -q basicsr facexlib gfpgan realesrgan cloudinary requests yt-dlp
!git clone https://github.com/xinntao/Real-ESRGAN.git || echo 'already cloned'
%cd /content/Real-ESRGAN
!pip install -q -r requirements.txt
!python setup.py develop

# Pre-download the x4plus general (non-anime) weights so first inference doesn't stall.
import os
os.makedirs('weights', exist_ok=True)
if not os.path.exists('weights/RealESRGAN_x4plus.pth'):
    !wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P weights/
print('weights:')
!ls -la weights/

# basicsr ships a degradations.py that imports from torchvision.transforms.functional_tensor,
# which was removed in newer torchvision. If the import breaks on Colab, patch it inline:
import subprocess, re, pathlib
for p in pathlib.Path('/usr/local/lib').glob('python*/dist-packages/basicsr/data/degradations.py'):
    s = p.read_text()
    if 'functional_tensor' in s:
        p.write_text(s.replace('torchvision.transforms.functional_tensor',
                                'torchvision.transforms.functional'))
        print('patched', p)
print('Setup complete')

## Cell 3 — Mount Google Drive + configure

Set `MANIFEST['source_url']` to the YouTube URL of the full game (or a Cloudinary URL).
Set the clips list using `startTime`/`endTime` from your detection result.

Get the manifest from Railway:
```
curl https://jersey-detection-production-d8d8.up.railway.app/analyze-jobs/<JOB_ID>
```

Cloudinary credentials come from <https://cloudinary.com/console>. Account `dc33vjyyv` is the Clipt cloud.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, cloudinary

# === CONFIGURE THESE ===
CLOUDINARY_CLOUD_NAME = 'dc33vjyyv'
CLOUDINARY_API_KEY    = ''      # paste from cloudinary.com/console
CLOUDINARY_API_SECRET = ''      # paste from cloudinary.com/console

MANIFEST = {
    'source_url': 'https://www.youtube.com/watch?v=x0bZnfVHDx4',
    'clips': [
        # paste from detection result — at minimum {id, startTime, endTime}
        # {'id': 1, 'startTime': 1686, 'endTime': 1701, 'label': 'snap-run-tackle'},
        # {'id': 2, 'startTime': 5300, 'endTime': 5315, 'label': 'play-action'},
    ],
}

assert MANIFEST['clips'], 'Set MANIFEST["clips"] before continuing'
assert CLOUDINARY_API_KEY and CLOUDINARY_API_SECRET, 'Set Cloudinary credentials'

cloudinary.config(
    cloud_name=CLOUDINARY_CLOUD_NAME,
    api_key=CLOUDINARY_API_KEY,
    api_secret=CLOUDINARY_API_SECRET,
)
print(f'Configured. Will process {len(MANIFEST["clips"])} clips.')

## Cell 4 — Download source video + cut clip segments

Downloads once at ≤720p (yt-dlp) so we don't pay the bandwidth twice.
If the source URL is direct HTTP, falls back to streamed `requests` download.
Each clip is then cut with ffmpeg using `-ss/-to` and re-encoded with x264 CRF 18 to keep quality before upscaling.

In [ ]:
import subprocess, os, json, time
from pathlib import Path

WORK = Path('/content/work'); WORK.mkdir(exist_ok=True)
(WORK / 'segments').mkdir(exist_ok=True)
(WORK / 'upscaled').mkdir(exist_ok=True)

src_url = MANIFEST['source_url']
src_path = WORK / 'source.mp4'

if not src_path.exists():
    if 'youtube.com' in src_url or 'youtu.be' in src_url:
        print(f'yt-dlp ← {src_url}')
        !yt-dlp -f 'bestvideo[height<=720][ext=mp4]+bestaudio[ext=m4a]/best[height<=720][ext=mp4]/best' \
            --merge-output-format mp4 \
            -o '{src_path}' '{src_url}'
    else:
        print(f'curl ← {src_url[:80]}')
        !curl -sSL '{src_url}' -o '{src_path}'

size_mb = src_path.stat().st_size // 1024 // 1024
probe = subprocess.run(
    ['ffprobe', '-v', 'error', '-select_streams', 'v:0',
     '-show_entries', 'stream=width,height,duration',
     '-of', 'csv=p=0', str(src_path)],
    capture_output=True, text=True,
)
print(f'Source: {size_mb}MB — {probe.stdout.strip()}')

for clip in MANIFEST['clips']:
    out = WORK / 'segments' / f'clip_{clip["id"]:02d}_{int(clip["startTime"])}s.mp4'
    cmd = ['ffmpeg', '-y',
           '-ss', str(clip['startTime']),
           '-to', str(clip['endTime']),
           '-i', str(src_path),
           '-c:v', 'libx264', '-preset', 'fast', '-crf', '18',
           '-c:a', 'aac', str(out)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if out.exists():
        mb = out.stat().st_size // 1024 // 1024
        print(f'clip {clip["id"]} ({clip["startTime"]}s–{clip["endTime"]}s): {mb}MB')
    else:
        print(f'clip {clip["id"]} FAILED: {r.stderr[-200:]}')

## Cell 5 — Upscale each clip with Real-ESRGAN

`--outscale 2` clamps the natively-4x model to 2x (360p→720p; 4x is unnecessary and 4x more expensive).
`--num_process_per_gpu 2` parallelizes on a single A100.
`--fp32` keeps text-edge fidelity (jersey numbers); fp16 can blur small text.
First clip will print full timing so you can calibrate.

In [ ]:
import time, subprocess, glob, json

upscaled = []
wall_t0 = time.perf_counter()
for clip in MANIFEST['clips']:
    seg = f'/content/work/segments/clip_{clip["id"]:02d}_{int(clip["startTime"])}s.mp4'
    out_dir = '/content/work/upscaled'
    if not os.path.exists(seg):
        print(f'clip {clip["id"]}: segment missing, skip')
        continue
    t0 = time.perf_counter()
    print(f'\n=== upscaling clip {clip["id"]} ===')
    r = subprocess.run([
        'python', '/content/Real-ESRGAN/inference_realesrgan_video.py',
        '-i', seg,
        '-o', out_dir,
        '-n', 'RealESRGAN_x4plus',
        '--outscale', '2',
        '--num_process_per_gpu', '2',
        '--fp32',
        '--suffix', '720p',
    ], capture_output=True, text=True, cwd='/content/Real-ESRGAN')
    elapsed = time.perf_counter() - t0

    matches = sorted(glob.glob(f'{out_dir}/clip_{clip["id"]:02d}_*720p*.mp4'))
    if matches:
        out_path = matches[-1]
        mb = os.path.getsize(out_path) // 1024 // 1024
        probe = subprocess.run(
            ['ffprobe', '-v', 'error', '-select_streams', 'v:0',
             '-show_entries', 'stream=width,height',
             '-of', 'csv=p=0', out_path],
            capture_output=True, text=True,
        )
        print(f'  {probe.stdout.strip()} {mb}MB in {elapsed:.0f}s')
        upscaled.append({'clip': clip, 'path': out_path, 'elapsed': elapsed})
    else:
        print(f'  FAILED in {elapsed:.0f}s')
        print('  stdout:', r.stdout[-300:])
        print('  stderr:', r.stderr[-300:])

print(f'\nTotal: {len(upscaled)}/{len(MANIFEST["clips"])} clips upscaled in {time.perf_counter()-wall_t0:.0f}s')

## Cell 6 — Upload to Cloudinary + write manifest

Each upscaled clip lands in folder `clipt-clips-720p/` with a stable public_id keyed on clip id and start time. Re-runs overwrite by default.

The output manifest can be pasted directly into the next detection or render call — see the README at `colab/README.md` in this repo.

In [ ]:
import cloudinary.uploader, json

results = []
for item in upscaled:
    clip = item['clip']
    path = item['path']
    public_id = f'clip_{clip["id"]:02d}_{int(clip["startTime"])}s_720p'
    print(f'uploading {public_id} …')
    r = cloudinary.uploader.upload_large(
        path,
        resource_type='video',
        folder='clipt-clips-720p',
        public_id=public_id,
        overwrite=True,
    )
    results.append({
        'clip_id': clip['id'],
        'startTime': clip['startTime'],
        'endTime': clip['endTime'],
        'label': clip.get('label', ''),
        'cloudinary_url': r.get('secure_url', ''),
        'width': r.get('width'),
        'height': r.get('height'),
    })
    print(f'  → {r.get("secure_url", "")[:90]}')

manifest_out = {
    'source_url': MANIFEST['source_url'],
    'upscaled_clips': results,
}
out_path = '/content/drive/MyDrive/clipt_upscaled_manifest.json'
with open(out_path, 'w') as f:
    json.dump(manifest_out, f, indent=2)

print('\n=== UPSCALED CLIP URLS ===')
for r in results:
    print(f'clip {r["clip_id"]} ({r["startTime"]}s–{r["endTime"]}s): {r["cloudinary_url"]}')
print(f'\nManifest saved: {out_path}')

## Next steps

1. Re-run detection or render on the upscaled clips by passing the Cloudinary URLs into the Railway detection pipeline. Higher input resolution should improve EasyOCR jersey hits and tighten the frame-diff motion signal.
2. If jersey OCR is still weak after upscaling, run `clipt_jersey_ocr_training.ipynb` to fine-tune PARSeq on these crops.